# DeepCritical Gradio Demo

This notebook shows how to run a lightweight DeepCritical research agent with a Gradio interface. It is designed to run in the Colab free tier by default, falling back to built-in mock tools when API keys are unavailable.

## 1) Install dependencies

* On Colab, the cell installs the latest `dev` branch of DeepCritical.
* Locally, it installs the current repository in editable mode.
* Gradio is installed in both cases for the UI.

In [ ]:

import sys, os, subprocess


# Install dependencies. Using subprocess keeps the cell compatible with both IPython
# notebooks and plain Python execution.
def _pip_install(args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + args)


common = ["gradio>=4.44.0", "beautifulsoup4>=4.12.0"]

if "google.colab" in sys.modules:
    _pip_install(common + ["git+https://github.com/DeepCritical/DeepCritical.git@dev"])
else:
    _pip_install(common)
    _pip_install(["-e", "."])


## 2) Import and register DeepCritical tools

Importing `DeepResearch.src.tools` registers every tool with the internal registry, which is required before executing any plans. The notebook also defines helpers for building configs and summarizing tool history.

In [ ]:

import json
import os
from typing import Any, Tuple

import gradio as gr
from omegaconf import OmegaConf

import DeepResearch
from DeepResearch.agents import ExecutionHistory, ExecutorAgent, ParserAgent, PlannerAgent
from DeepResearch.app import run_graph
from DeepResearch.src.tools import deepsearch_tools, mock_tools, workflow_tools
from DeepResearch.src.tools.base import registry


def build_demo_config(question: str) -> OmegaConf:
    "Create a minimal DictConfig for running the DeepCritical graph."
    return OmegaConf.create(
        {
            "question": question,
            "retries": 1,
            "flows": {"deepsearch": {"enabled": True}},
        }
    )


def summarize_history(history: ExecutionHistory) -> list[dict[str, Any]]:
    "Return a compact summary of tool executions for display."
    return [
        {
            "tool": item.get("tool"),
            "success": item.get("success"),
            "error": item.get("error"),
            "duration": round(item.get("execution_time", 0.0), 3),
        }
        for item in history.items
    ]


## 3) Offline-friendly research runner

The helper below executes a research plan with the existing `ParserAgent`, `PlannerAgent`, and `ExecutorAgent`. If no API keys are present, the agents gracefully fall back to deterministic placeholder tools (e.g., rewrite, web search, summarize, references).

In [ ]:

def run_offline_research(question: str) -> Tuple[str, dict[str, Any], list[dict[str, Any]]]:
    "Run DeepCritical with mock-friendly tools and return result, bag, and history."

    parser = ParserAgent()
    planner = PlannerAgent()

    parsed = parser.parse(question)
    plan = planner.plan(parsed)

    history = ExecutionHistory()
    executor = ExecutorAgent(retries=1)
    bag = executor.run_plan(plan, history)

    final = (
        bag.get("finalize.final")
        or bag.get("final")
        or bag.get("references.answer_with_refs")
        or bag.get("summarize.summary")
        or "No answer produced."
    )

    return final, bag, summarize_history(history)


## 4) Full graph execution (optional API key)

To use a hosted model, provide an API key that matches the provider prefix:
* `openai:gpt-4o-mini` → `OPENAI_API_KEY`
* `anthropic:claude-sonnet-4-0` → `ANTHROPIC_API_KEY`

Without a key, the graph still runs with the built-in mock tools.

In [ ]:

def run_graph_research(question: str, model: str, api_key: str | None = None) -> str:
    # Set provider-specific keys so pydantic-ai can find credentials if available.
    if api_key:
        if model.startswith("openai"):
            os.environ["OPENAI_API_KEY"] = api_key
        elif model.startswith("anthropic"):
            os.environ["ANTHROPIC_API_KEY"] = api_key

    cfg = build_demo_config(question)
    return run_graph(question, cfg)


## 5) Gradio interface

Select an execution mode, optionally enter an API key, and click **Run Deep Research**. The interface returns the answer, the tool output bag, and a compact execution log.

In [ ]:

def launch_demo():
    modes = ["Offline mock tools", "Full graph (with optional API key)"]
    models = ["openai:gpt-4o-mini", "anthropic:claude-sonnet-4-0"]

    with gr.Blocks() as demo:
        gr.Markdown(
            '''### DeepCritical Research Agent
Enter a research question and choose how to run it.
* **Offline mock tools** use built-in deterministic tools.
* **Full graph** can use real APIs when keys are provided.
'''
        )

        with gr.Row():
            question = gr.Textbox(
                label="Research question",
                lines=2,
                value="How do transformer models handle long context?",
            )
        with gr.Row():
            mode = gr.Radio(modes, value=modes[0], label="Execution mode")
            model = gr.Dropdown(models, value=models[0], label="Preferred model (graph mode)")
            api_key = gr.Textbox(label="API key (optional)", type="password")

        answer = gr.Markdown(label="Answer")
        bag_out = gr.JSON(label="Tool output bag")
        history_out = gr.JSON(label="Execution history")

        def _run(question: str, mode: str, model: str, api_key: str):
            if not question.strip():
                return "Please provide a question.", {}, []

            if mode == modes[0]:
                final, bag, history = run_offline_research(question)
                return final, bag, history

            result = run_graph_research(question, model=model, api_key=api_key or None)
            return result, {"result": result}, []

        run_btn = gr.Button("Run Deep Research")
        run_btn.click(
            _run,
            inputs=[question, mode, model, api_key],
            outputs=[answer, bag_out, history_out],
        )

    return demo


# Uncomment the line below when running interactively in Colab or a notebook.
# launch_demo().launch()


## 6) Quick smoke test

The cell below runs the offline pipeline once so you can verify the setup without launching Gradio.

In [ ]:

            if __name__ == "__main__":
                sample_answer, sample_bag, sample_history = run_offline_research("What is DeepCritical?")
                print("Sample answer:
", sample_answer)
                print("
Outputs:
", json.dumps(sample_bag, indent=2))
                print("
History:
", json.dumps(sample_history, indent=2))
